# End-to-End Machine Learning Project: Linear Regression on a Dirty Kaggle Housing Dataset

**Chosen dataset:** Kaggle - **Messy House Price Sample datasets**

This notebook demonstrates a practical end-to-end analytics workflow:
- inspect the data
- identify dirty-data problems
- clean the data with Pandas
- prepare features and target
- split train/test
- train a **Linear Regression** model
- evaluate with **MAE, MSE, and R²**
- create visualizations
- interpret the results


## Notes Before Running
1. Download the dataset from Kaggle.
2. In Colab, upload the CSV file when prompted.
3. This notebook is written to be flexible with messy housing-price CSV files.
4. The code will try to detect the target column automatically. If needed, change `target_col` manually.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print("Uploaded file:", file_name)


In [ ]:
df = pd.read_csv(file_name)
df.head()


In [ ]:
print("Shape:", df.shape)
print()
print("Columns:")
print(df.columns.tolist())
print()
print("Info:")
print(df.info())
print()
print("Describe:")
display(df.describe(include="all"))


In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print()
print("Duplicate rows:", df.duplicated().sum())


In [ ]:
# make column names easier to work with
clean_cols = []
for col in df.columns:
    new_col = col.strip().lower().replace(" ", "_")
    clean_cols.append(new_col)

df.columns = clean_cols
print(df.columns.tolist())


In [ ]:
# replace common dirty values with real missing values
for col in df.columns:
    df[col] = df[col].replace(["?", "NA", "N/A", "na", "n/a", "None", "none", "null", "NULL", "", " ", "unknown", "Unknown"], np.nan)


In [ ]:
# remove duplicate rows
df = df.drop_duplicates()
print("New shape after removing duplicates:", df.shape)


In [ ]:
# try to detect the target column automatically
possible_targets = ["price", "saleprice", "selling_price", "house_price", "target"]
target_col = None

for col in possible_targets:
    if col in df.columns:
        target_col = col
        break

print("Detected target column:", target_col)


In [ ]:
# if target_col is still None, set it manually here
# example: target_col = "price"
target_col = target_col

if target_col is None:
    raise ValueError("Target column not found automatically. Please set target_col manually.")


In [ ]:
# clean the target column if it has symbols or commas
if df[target_col].dtype == "object":
    df[target_col] = df[target_col].astype(str)
    df[target_col] = df[target_col].str.replace(",", "", regex=False)
    df[target_col] = df[target_col].str.replace("$", "", regex=False)
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

print(df[target_col].head())


In [ ]:
# drop rows where the target is missing
df = df.dropna(subset=[target_col])
print("Shape after dropping missing target rows:", df.shape)


In [ ]:
# separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# try to convert object columns that actually contain numbers
for col in X.columns:
    if X[col].dtype == "object":
        temp = X[col].astype(str).str.replace(",", "", regex=False).str.replace("$", "", regex=False)
        converted = pd.to_numeric(temp, errors="coerce")
        not_null_count = converted.notna().sum()

        if not_null_count > 0 and not_null_count >= len(X[col]) * 0.6:
            X[col] = converted


In [ ]:
# identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print()
print("Categorical columns:", categorical_cols)


In [ ]:
# preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)


In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("MSE:", mse)
print("R2 Score:", r2)


In [ ]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
results.head(10)


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted House Prices")
plt.show()


In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(6, 5))
plt.hist(residuals, bins=20)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.show()


In [ ]:
# show top numeric correlations with price if possible
corr_df = df.select_dtypes(include=[np.number]).corr()

if target_col in corr_df.columns:
    corr_with_target = corr_df[target_col].sort_values(ascending=False)
    print(corr_with_target)


## Interpretation Guide
Use the generated output values when presenting results.

Suggested explanation:
- The dataset was dirty because it had missing values, duplicate rows, and inconsistent values.
- We cleaned the data using Pandas by replacing invalid placeholders with `NaN`, removing duplicates, and imputing missing values.
- We used **Linear Regression** because the target variable is numeric house price.
- We evaluated the model using **MAE**, **MSE**, and **R² Score**.
- A lower MAE and MSE means the prediction errors are smaller.
- An R² score closer to 1 means the model explains more of the variation in house prices.
- The Actual vs Predicted plot shows how close predictions are to real prices.
- The residual plot helps us see whether the errors are spread out normally or if patterns remain.
